In [1]:
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from torch.nn.functional import softmax
from torch.optim import AdamW
from torch.utils.data import DataLoader

from internal.data_types import HistologyDataset
from internal.nn.model import train_one_epoch, validate
from internal.persistence_manager import PersistenceManager

data = PersistenceManager.load_dataset()
test_df = data.test_df
train_df = data.train_df
train_transforms = data.train_transforms
val_test_transforms = data.val_test_transforms
idx2label = data.idx2label

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cuda_is_available = torch.cuda.is_available()
print(f'Using device: {device}')

Arrays and scalers loaded successfully from: /home/andre/university/AN2DL-Challenge-2/notebooks/processed/dataset.joblib
Using device: cuda


In [2]:
N_FOLDS = data.num_K_folds
BATCH_SIZE = 4
PRETRAINED_MODEL = "tf_efficientnetv2_s.in21k"
N_CLASSES = 4 # number of classes in the dataset (labels)

for fold in range(N_FOLDS):
    print(f"\n========== Fold {fold} ==========")

    train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
    val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

    train_dataset = HistologyDataset(train_df_split, transforms=train_transforms, is_train=True)
    val_dataset   = HistologyDataset(val_df_split,   transforms=val_test_transforms, is_train=True)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=4, pin_memory=cuda_is_available)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=cuda_is_available)

    # --- create fresh model for this fold ---
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=True,
        num_classes=N_CLASSES
    ).to(device)

    # --- Stage 1: freeze backbone, train classifier head ---
    print("\n--- Stage 1: Training classifier head ---")

    # --- 1.1. freeze feature extractor layers ---
    for param in model.parameters():
        param.requires_grad = False

    # 2) unfreeze classifier head (EffNetV2 uses .classifier)
    for param in model.classifier.parameters():
        param.requires_grad = True

    # --- 1.2. define loss, optimizer, scheduler ---
    criterion = nn.CrossEntropyLoss()
    head_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=10
    )

    # --- 1.3. train for several epochs ---
    EPOCHS = 8
    best_f1 = 0.0
    best_state = None
    for epoch in range(1, EPOCHS+1):
        print(f"\nEpoch {epoch}/{EPOCHS}")
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc, val_f1 = validate(
            model, val_loader, criterion, device
        )
        scheduler.step()
        print(
            f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
            f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
        )
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = model.state_dict().copy()
            torch.save(best_state, f"best_effv2_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
            print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

    # --- Stage 2: unfreeze whole model, fine-tune ---
    print("\n--- Stage 2: Fine-tuning entire model ---")

    # --- 2.1. unfreeze entire model ---
    for param in model.parameters():
        param.requires_grad = True

    # --- 2.2. define loss, optimizer, scheduler ---
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=15
    )

    # --- 2.3. mild class weights ---
    class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32) # 445 LumB, 414 LumA, 397 Her2, 156 TN
    class_weights = (class_counts.sum() / class_counts)
    class_weights = class_weights / class_weights.mean()
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

    # --- 2.4. train for several epochs ---
    EPOCHS = 15
    best_f1 = 0.0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc, val_f1 = validate(
            model, val_loader, criterion, device
        )
        scheduler.step()
        print(
            f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
            f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
        )
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = model.state_dict().copy()
            torch.save(best_state, f"best_effv2_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
            print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

    # --- save model for this fold ---
    torch.save(model.state_dict(), f"effv2_s_fold{fold}.pth")


========== Fold 0 ==========

Epoch 1/8


[] t_loss=4.0917 | F1(macro)=0.2446 | Acc=0.2817


Train  loss=4.0917 acc=0.2817 f1=0.2446 | Val loss=5.9574 acc=0.2756 f1=0.2133
  🔥 New best F1: 0.2133 – model saved.

Epoch 2/8


[] t_loss=3.4673 | F1(macro)=0.2693 | Acc=0.3056


Train  loss=3.4673 acc=0.3056 f1=0.2693 | Val loss=5.0011 acc=0.2792 f1=0.2224
  🔥 New best F1: 0.2224 – model saved.

Epoch 3/8


...

Train  loss=0.7695 acc=0.6850 f1=0.6811 | Val loss=1.8464 acc=0.3475 f1=0.3297

Epoch 14/15


[] t_loss=0.7530 | F1(macro)=0.6976 | Acc=0.6982


Train  loss=0.7530 acc=0.6982 f1=0.6976 | Val loss=1.8750 acc=0.3511 f1=0.3381

Epoch 15/15


[] t_loss=0.7297 | F1(macro)=0.7040 | Acc=0.7009


Train  loss=0.7297 acc=0.7009 f1=0.7040 | Val loss=1.8977 acc=0.3652 f1=0.3446
  🔥 New best F1: 0.3446 – model saved.


In [3]:
# ensemble at inference time

test_dataset = HistologyDataset(test_df, transforms=val_test_transforms, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=4, pin_memory=cuda_is_available)

all_fold_probs = []  # list of [N, 4]
all_sample_indices = None

for fold in range(N_FOLDS):
    print(f"Inference with fold {fold} model")
    model = timm.create_model(PRETRAINED_MODEL, pretrained=False, num_classes=N_CLASSES).to(device)
    model.load_state_dict(torch.load(f"effv2_s_fold{fold}.pth", map_location=device))
    model.eval()

    fold_probs = []
    sample_indices_list = []

    with torch.no_grad():
        for imgs, sample_indices in test_loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = softmax(logits, dim=1).cpu().numpy()
            fold_probs.append(probs)
            if all_sample_indices is None:
                sample_indices_list.extend(sample_indices)

    fold_probs = np.concatenate(fold_probs, axis=0)
    all_fold_probs.append(fold_probs)

    if all_sample_indices is None:
        all_sample_indices = sample_indices_list

mean_probs = np.mean(all_fold_probs, axis=0)      # [N, 4]
pred_indices = mean_probs.argmax(axis=1)

pred_labels = [idx2label[int(i)] for i in pred_indices]
sample_index_with_ext = [f"{si}.png" if not si.endswith(".png") else si
                         for si in all_sample_indices]

submission_df = pd.DataFrame({
    "sample_index": sample_index_with_ext,
    "label": pred_labels
})
submission_df.to_csv("submission_5fold.csv", index=False)


Inference with fold 0 model
Inference with fold 1 model
Inference with fold 2 model
Inference with fold 3 model
Inference with fold 4 model
